# ✅ Citation Verifier

## What We're Building

A pipeline that checks whether a RAG-generated answer is **actually supported**
by the retrieved chunks. For each claim in the answer, we verify:

- ✅ **Supported**: the claim is directly stated in a retrieved chunk
- ⚠️ **Partially supported**: related info exists but doesn't fully confirm
- ❌ **Unsupported**: no evidence in the retrieved chunks (hallucination!)

## Why This Matters

RAG reduces hallucination compared to vanilla LLMs, but doesn't eliminate it.
The LLM can still:
- Blend retrieved facts with parametric knowledge
- Misinterpret numbers or dates from context
- Generate plausible-sounding claims not in any chunk

A citation verifier catches these failures **before the answer reaches the user**.

## Stack
- **LangChain** — RAG pipeline + verification chain
- **ChromaDB** — vector retrieval
- **Ollama** — local LLM for generation + verification

## Step 1 — Imports & Setup

In [ ]:
import json
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

In [ ]:
# A factual knowledge base we can verify against
knowledge = """# Solar System Facts

Mercury is the smallest planet and closest to the Sun. Its surface temperature
ranges from -180°C at night to 430°C during the day. It has no atmosphere and
no moons. Orbital period is 88 Earth days.

Venus is the hottest planet due to its thick CO2 atmosphere creating a runaway
greenhouse effect. Surface temperature averages 465°C. It rotates backward
(retrograde rotation). Venus has no moons.

Earth is the only known planet with liquid water on its surface. Average
temperature is 15°C. It has one natural satellite (the Moon). The Moon is
384,400 km away and affects Earth's tidal patterns.

Mars has the largest volcano in the solar system: Olympus Mons (21.9 km high).
It has two small moons: Phobos and Deimos. Average temperature is -65°C.
A Martian day (sol) is 24 hours 37 minutes.

Jupiter is the largest planet with a mass 318 times that of Earth. It has at
least 95 known moons, including the four Galilean moons: Io, Europa, Ganymede,
and Callisto. The Great Red Spot is a storm larger than Earth.

Saturn is famous for its ring system made of ice and rock particles. It has
146 known moons, the most of any planet. Titan, its largest moon, has a thick
atmosphere and liquid methane lakes.

Uranus rotates on its side with an axial tilt of 98 degrees. It has 28 known
moons and a faint ring system. Average temperature is -224°C.

Neptune has the strongest winds in the solar system, reaching 2,100 km/h.
It has 16 known moons, with Triton being the largest. Triton orbits in a
retrograde direction, suggesting it was captured. Neptune's orbital period
is 165 Earth years.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
docs = splitter.create_documents([knowledge])

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="citation_verify")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(docs)} chunks")

## Step 2 — RAG Answer Generator

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the provided context. Include specific
facts and numbers.

Context:
{context}

Question: {question}

Answer:"""
)

def generate_answer(question: str) -> tuple[str, list[str]]:
    """Standard RAG: retrieve + generate. Returns (answer, chunks)."""
    docs = retriever.invoke(question)
    chunks = [d.page_content for d in docs]
    context = "\n\n".join(chunks)
    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    return answer, chunks

print("RAG generator ready")

## Step 3 — Claim Extractor

Break the answer into **individual claims** that can be verified separately.

In [ ]:
extract_prompt = ChatPromptTemplate.from_template(
    """Extract individual factual claims from the answer below.
Each claim should be a single verifiable statement.

Rules:
- One claim per line
- Each claim should be self-contained
- Include specific numbers, names, and facts
- Skip filler phrases and opinions
- Number each claim

Answer:
{answer}

Extracted claims:"""
)


def extract_claims(answer: str) -> list[str]:
    """Extract verifiable claims from an answer."""
    chain = extract_prompt | llm | StrOutputParser()
    result = chain.invoke({"answer": answer})
    claims = []
    for line in result.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            cleaned = line.lstrip("0123456789.-) ").strip()
            if cleaned:
                claims.append(cleaned)
    return claims


# Test
test_answer = "Mars has two moons named Phobos and Deimos. It also has the largest volcano, Olympus Mons, which is 21.9 km high."
claims = extract_claims(test_answer)
print("Test claims:")
for i, c in enumerate(claims, 1):
    print(f"  {i}. {c}")

## Step 4 — Citation Verifier

For each claim, check if it's supported by the retrieved chunks.

In [ ]:
verify_prompt = ChatPromptTemplate.from_template(
    """Determine if the following CLAIM is supported by the EVIDENCE chunks.

CLAIM: {claim}

EVIDENCE:
{evidence}

Respond with EXACTLY one of:
- SUPPORTED: The evidence directly states or clearly implies this claim
- PARTIAL: The evidence has related information but doesn't fully confirm
- UNSUPPORTED: No evidence supports this claim

Then briefly explain why in one sentence.

Verdict:"""
)


def verify_claim(claim: str, evidence_chunks: list[str]) -> dict:
    """Verify a single claim against evidence chunks."""
    evidence = "\n\n---\n\n".join(evidence_chunks)
    chain = verify_prompt | llm | StrOutputParser()
    result = chain.invoke({"claim": claim, "evidence": evidence})

    # Parse verdict
    first_line = result.strip().split("\n")[0].upper()
    if "UNSUPPORTED" in first_line:
        verdict = "UNSUPPORTED"
    elif "PARTIAL" in first_line:
        verdict = "PARTIAL"
    else:
        verdict = "SUPPORTED"

    return {"claim": claim, "verdict": verdict, "explanation": result.strip()}


def verify_answer(question: str, answer: str, chunks: list[str]) -> list[dict]:
    """Full verification pipeline: extract claims → verify each."""
    claims = extract_claims(answer)
    results = []
    for claim in claims:
        result = verify_claim(claim, chunks)
        results.append(result)
    return results


print("Citation verifier ready")

## Step 5 — Full Pipeline: Generate → Verify → Report

In [ ]:
ICONS = {"SUPPORTED": "✅", "PARTIAL": "⚠️", "UNSUPPORTED": "❌"}


def rag_with_verification(question: str) -> None:
    """Generate an answer and verify every claim."""
    print(f"❓ {question}")
    print("=" * 60)

    # Generate
    answer, chunks = generate_answer(question)
    print(f"\n📝 Answer:\n{answer}")

    # Verify
    print(f"\n🔍 Verifying claims...")
    results = verify_answer(question, answer, chunks)

    # Report
    print(f"\n📊 Verification Report:")
    print("-" * 60)
    for i, r in enumerate(results, 1):
        icon = ICONS.get(r["verdict"], "?")
        print(f"  {icon} Claim {i}: {r['claim']}")
        print(f"     → {r['verdict']}")
        print()

    # Summary
    total = len(results)
    supported = sum(1 for r in results if r["verdict"] == "SUPPORTED")
    partial = sum(1 for r in results if r["verdict"] == "PARTIAL")
    unsupported = sum(1 for r in results if r["verdict"] == "UNSUPPORTED")

    score = (supported + 0.5 * partial) / total if total > 0 else 0
    print(f"{'='*60}")
    print(f"  Groundedness Score: {score:.0%}")
    print(f"  ✅ Supported: {supported}/{total}")
    print(f"  ⚠️  Partial: {partial}/{total}")
    print(f"  ❌ Unsupported: {unsupported}/{total}")

    if unsupported > 0:
        print(f"\n  ⚠️  WARNING: {unsupported} claim(s) may be hallucinated!")
    print("=" * 60)

In [ ]:
# Test 1: Question with good retrieval
rag_with_verification("Tell me about Mars — its moons, temperature, and notable features.")

In [ ]:
# Test 2: Question that might trigger hallucination
rag_with_verification("Which planets have rings and moons? Compare Jupiter and Saturn.")

In [ ]:
# Test 3: Question about temperature extremes
rag_with_verification("What are the temperature extremes across the solar system?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Claim extraction** | Breaks answers into verifiable atomic statements |
| **Per-claim verification** | Checks each claim against evidence independently |
| **Three-level verdicts** | SUPPORTED / PARTIAL / UNSUPPORTED classification |
| **Groundedness score** | Quantifies how well-grounded the answer is |
| **Hallucination detection** | Flags claims not supported by any chunk |

## 🔧 Customization Ideas

- **NLI model**: Use a fine-tuned natural language inference model instead of LLM
- **Auto-fix**: If claims are unsupported, regenerate the answer without them
- **Chunk attribution**: Link each supported claim to its specific source chunk
- **Confidence scores**: Add numerical confidence instead of 3-level verdict